In [2]:
import os
import time
import random
import warnings

import torch
import torch.nn.functional as F

from transformers import AutoModel, AutoProcessor

from torch.export import export

from executorch import version
from executorch.runtime import Runtime
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner


random.seed(0)
warnings.simplefilter("ignore", FutureWarning)
print("Executorch version: ", version.__version__)

Executorch version:  1.1.0


In [3]:
SUFFIX = "_xnnpack_fp32"
EXPORT_DIR = "exports"
MODEL_PATH: dict[str, str] = {}

processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")


class ClipVisionModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        model = AutoModel.from_pretrained("openai/clip-vit-base-patch32")
        self.vision_model = model.vision_model
        self.visual_projection = model.visual_projection

    def forward(self, pixel_values):
        model_output = self.vision_model(pixel_values)
        pooler_output = model_output.pooler_output
        projected_output = self.visual_projection(pooler_output)
        return F.normalize(projected_output, p=2, dim=1)

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [4]:
print("Exporting model...")

model = ClipVisionModel().eval()
inputs = (processor(images=torch.randint(0, 256, (1, 3, 224, 224)), return_tensors="pt")["pixel_values"],)

program = to_edge_transform_and_lower(export(model, inputs), partitioner=[XnnpackPartitioner()])
program = program.to_executorch()

os.makedirs(EXPORT_DIR, exist_ok=True)
with open(path := f"{EXPORT_DIR}/clip_vit_base_patch32_vision{SUFFIX}.pte", "wb") as f:
    MODEL_PATH["clip_vision"] = path
    f.write(program.buffer)

print(f"Exported {path}")

Exporting model...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 57782.38it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Exported exports/clip_vit_base_patch32_vision_xnnpack_fp32.pte


In [5]:
print("Verifying model...")

path = MODEL_PATH["clip_vision"]
pixel_values = processor(images=torch.randint(0, 256, (1, 3, 224, 224)), return_tensors="pt")["pixel_values"]

with torch.no_grad():
    expected = model(pixel_values)

runtime = Runtime.get()
et_program = runtime.load_program(path)
method = et_program.load_method("forward")
output, *_ = method.execute([pixel_values])  # type: ignore

torch.testing.assert_close(output, expected, rtol=1e-3, atol=1e-4)
print(
    f"Model size: {os.path.getsize(path) / 1024 / 1024:.2f} MB\n"
    "Comparison passed!"
)

[program.cpp:154] InternalConsistency verification requested but not available
[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:167] Number of efficient cores 4


Verifying model...
Model size: 335.32 MB
Comparison passed!


In [6]:
print("Benchmarking...")

path = MODEL_PATH["clip_vision"]
runtime = Runtime.get()
et_program = runtime.load_program(path)
method = et_program.load_method("forward")

pixel_values = processor(images=torch.randint(0, 256, (1, 3, 224, 224)), return_tensors="pt")["pixel_values"]

tik = time.perf_counter()
for _ in range(100):
    method.execute([pixel_values])  # type: ignore
tok = time.perf_counter()

print(
    f"{path}\n"
    f"Inference time (avg over 100 runs): {(tok - tik) / 100 * 1000:.2f} ms\n"
    f"Model size: {os.path.getsize(path) / 1024 / 1024:.2f} MB\n"
)

[program.cpp:154] InternalConsistency verification requested but not available


Benchmarking...
exports/clip_vit_base_patch32_vision_xnnpack_fp32.pte
Inference time (avg over 100 runs): 48.52 ms
Model size: 335.32 MB

